# Demo: end-to-end inference


# Demo: end-to-end inference (Tasks 1-3)

Loads each task's trained checkpoint (from `results/checkpoints/`) and runs
one real inference example, end to end. Run the earlier `python -m src.train
--task N` commands first so the checkpoints referenced here actually exist.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import torch
import yaml

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

CKPT_DIR = "../" + cfg["paths"]["checkpoints_dir"]
print("Loading checkpoints from:", CKPT_DIR)

## Task 1: BERT tag classifier (MagnaTagATune)

Give it a few tags as a pseudo-caption (the same format the model was trained
on -- see `src/datasets.py`'s docstring for why MagnaTagATune uses this
instead of real captions) and see what additional tags it predicts.

In [ ]:
from bert_encoder import BERTEncoder, TagClassifierHead
from transformers import AutoTokenizer

ckpt1 = torch.load(os.path.join(CKPT_DIR, "task1_model.pt"), map_location="cpu", weights_only=False)
tag_names = ckpt1["tag_names"]

encoder1 = BERTEncoder(model_name=cfg["model"]["bert"]["name"], freeze_base=cfg["model"]["bert"]["freeze_base"])
head1 = TagClassifierHead(hidden_dim=encoder1.hidden_dim, num_tags=len(tag_names))
encoder1.load_state_dict(ckpt1["encoder"])
head1.load_state_dict(ckpt1["head"])
encoder1.eval(); head1.eval()

tokenizer1 = AutoTokenizer.from_pretrained(cfg["model"]["bert"]["name"])

sample_text = "guitar, quiet"

encoded = tokenizer1([sample_text], padding=True, truncation=True,
                      max_length=cfg["text"]["max_length"], return_tensors="pt")
with torch.no_grad():
    cls_embedding, _ = encoder1(encoded["input_ids"], encoded["attention_mask"])
    probs = torch.sigmoid(head1(cls_embedding))[0]

predicted = [(tag_names[i], round(p.item(), 3)) for i, p in enumerate(probs) if p > 0.5]
predicted.sort(key=lambda x: -x[1])

print(f"Input text: {sample_text!r}")
print("Predicted tags (prob > 0.5):")
for tag, prob in predicted:
    print(f"  {tag}: {prob}")

## Task 2: GNN genre classifier (FMA)

Builds a real segment graph from an audio file (the same
`audio_features.py` -> `graph_builder.py` pipeline used in training) and
predicts its genre.

In [ ]:
from gnn_model import GraphEncoder, GraphTagHead
from datasets import fma_track_path, _track_to_graph
from torch_geometric.data import Batch

ckpt2 = torch.load(os.path.join(CKPT_DIR, "task2_gnn_model.pt"), map_location="cpu", weights_only=False)
genre_names_2 = ckpt2["genre_names"]

encoder2 = GraphEncoder(
    in_dim=ckpt2["in_dim"], hidden_dim=cfg["model"]["gnn"]["hidden_dim"],
    num_layers=cfg["model"]["gnn"]["num_layers"], encoder_type=cfg["model"]["gnn"]["type"],
    dropout=cfg["model"]["gnn"]["dropout"],
)
head2 = GraphTagHead(hidden_dim=encoder2.hidden_dim, num_classes=len(genre_names_2))
encoder2.load_state_dict(ckpt2["encoder"])
head2.load_state_dict(ckpt2["head"])
encoder2.eval(); head2.eval()

fma_cfg = cfg["dataset"]["fma"]
audio_dir_2 = "../" + fma_cfg["audio_dir"]

graph = _track_to_graph(
    sample_track_id, audio_dir_2,
    sample_rate=cfg["dataset"]["sample_rate"], hop_length=cfg["audio_features"]["hop_length"],
    segment_seconds=cfg["dataset"]["segment_seconds"],
    similarity_threshold=cfg["graph"]["segment_similarity_threshold"],
    feature_type="chroma", n_mels=cfg["audio_features"]["n_mels"], n_chroma=cfg["audio_features"]["n_chroma"],
)
batch = Batch.from_data_list([graph])

with torch.no_grad():
    _, graph_embedding = encoder2(batch.x, batch.edge_index, batch.batch)
    probs = torch.sigmoid(head2(graph_embedding))[0]

pred_idx = probs.argmax().item()
print(f"Track: {fma_track_path(sample_track_id, audio_dir_2)}")
print(f"Predicted genre: {genre_names_2[pred_idx]} (confidence {probs[pred_idx]:.3f})")
print("All genre probabilities:")
for name, p in sorted(zip(genre_names_2, probs.tolist()), key=lambda x: -x[1]):
    print(f"  {name}: {p:.3f}")

## Task 3: GNN-BERT cross-attention fusion (FMA)

Combines a track's audio graph with a short tag-based text hint to predict
genre -- shows how much the text nudges the prediction versus Task 2's
graph-only model above.

In [ ]:
from fusion_model import GNNBERTFusionModel

ckpt3 = torch.load(os.path.join(CKPT_DIR, "task3_cross_attention_model.pt"), map_location="cpu", weights_only=False)
genre_names_3 = ckpt3["genre_names"]

graph_encoder_3 = GraphEncoder(
    in_dim=ckpt3["in_dim"], hidden_dim=cfg["model"]["gnn"]["hidden_dim"],
    num_layers=cfg["model"]["gnn"]["num_layers"], encoder_type=cfg["model"]["gnn"]["type"],
    dropout=cfg["model"]["gnn"]["dropout"],
)
text_encoder_3 = BERTEncoder(model_name=cfg["model"]["bert"]["name"], freeze_base=cfg["model"]["bert"]["freeze_base"])
model3 = GNNBERTFusionModel(
    graph_encoder=graph_encoder_3, text_encoder=text_encoder_3,
    graph_dim=graph_encoder_3.hidden_dim, text_dim=text_encoder_3.hidden_dim,
    num_tags=len(genre_names_3), predict_emotion=False,
)
model3.load_state_dict(ckpt3["model"])
model3.eval()

sample_text_3 = "upbeat, synth"
tokenizer3 = AutoTokenizer.from_pretrained(cfg["model"]["bert"]["name"])
encoded3 = tokenizer3([sample_text_3], padding=True, truncation=True,
                       max_length=cfg["text"]["max_length"], return_tensors="pt")

with torch.no_grad():
    out3 = model3(batch, encoded3["input_ids"], encoded3["attention_mask"], return_attention=True)
    probs3 = torch.sigmoid(out3["tag_logits"])[0]

pred_idx3 = probs3.argmax().item()
print(f"Track: {fma_track_path(sample_track_id, audio_dir_2)}")
print(f"Text hint: {sample_text_3!r}")
print(f"Predicted genre: {genre_names_3[pred_idx3]} (confidence {probs3[pred_idx3]:.3f})")

tokens = tokenizer3.convert_ids_to_tokens(encoded3["input_ids"][0].tolist())
attn = out3["attention_weights"][0].tolist()
print("\nWhich text tokens the model attended to most:")
for tok, w in sorted(zip(tokens, attn), key=lambda x: -x[1])[:5]:
    if tok not in tokenizer3.all_special_tokens:
        print(f"  {tok}: {w:.4f}")